# Exploratory Analysis — Phase 1 & 2

A quick look at the cleaned data and the Phase 2 return calculations before they go to Tableau.

Run `python main.py` first so the files in `data/processed/` and `data/tableau/` exist.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
market = pd.read_csv(ROOT / 'data/tableau/tableau_market_data.csv', parse_dates=['Date'])
portfolio = pd.read_csv(ROOT / 'data/processed/clean_portfolio_data.csv', parse_dates=['Purchase_Date'])
market.head()

## 1. Dataset overview

In [ ]:
print('Shape:', market.shape)
print('Stocks:', market['Ticker'].nunique(), sorted(market['Ticker'].unique()))
print('Date range:', market['Date'].min().date(), 'to', market['Date'].max().date())
print('Sectors:', sorted(market['Sector'].unique()))
market.isna().sum()

The only blanks are `Daily_Return`, `Price_Change` and `Price_Change_Percentage` on each stock's **first day**: there is no previous price to compare with.

## 2. Closing price trend (same view as Tableau Sheet 1)

In [ ]:
prices = market.pivot(index='Date', columns='Ticker', values='Close')
ax = prices.plot(figsize=(10, 4), linewidth=1.5, title='Daily closing price (USD)')
ax.set_ylabel('USD'); plt.show()

## 3. Cumulative return — compounding check

`Cumulative_Return = product(1 + Daily_Return) - 1`. Its final value must equal `last adjusted close / first adjusted close - 1`.

In [ ]:
check = market.sort_values('Date').groupby('Ticker').agg(
    first_adj=('Adjusted_Close', 'first'), last_adj=('Adjusted_Close', 'last'),
    final_cumulative=('Cumulative_Return', 'last'))
check['direct_return'] = check['last_adj'] / check['first_adj'] - 1
check['difference'] = (check['final_cumulative'] - check['direct_return']).abs()
check.sort_values('final_cumulative', ascending=False).style.format(
    {'final_cumulative': '{:+.1%}', 'direct_return': '{:+.1%}', 'difference': '{:.1e}'})

In [ ]:
growth = market.pivot(index='Date', columns='Ticker', values='Cumulative_Return')
ax = (growth * 100).plot(figsize=(10, 4), linewidth=1.5, title='Cumulative total return (%)')
ax.axhline(0, color='grey', linewidth=0.8); ax.set_ylabel('%'); plt.show()

## 4. Daily return distribution

In [ ]:
summary = market.groupby('Ticker')['Daily_Return'].describe()[['mean', 'std', 'min', 'max']]
summary.style.format('{:+.2%}')

Standard deviation of daily returns is the starting point for the Phase 4 volatility analysis.

## 5. Sample portfolio allocation (SAMPLE / DEMO DATA)

In [ ]:
alloc = portfolio[['Ticker', 'Company', 'Quantity', 'Purchase_Price', 'Investment_Amount']].copy()
alloc['Share'] = alloc['Investment_Amount'] / alloc['Investment_Amount'].sum()
print(f"Total invested: ${alloc['Investment_Amount'].sum():,.2f}")
alloc.sort_values('Investment_Amount', ascending=False).style.format(
    {'Purchase_Price': '${:,.2f}', 'Investment_Amount': '${:,.2f}', 'Share': '{:.1%}'})

## Findings for the Phase 2 review

- Data is complete: 6 stocks × 437 trading days, no missing prices after cleaning.
- Caterpillar had the strongest total return over the period and Procter & Gamble the weakest (the only negative one).
- The compounded `Cumulative_Return` matches the direct price ratio, which confirms the calculation.
- Apple is the largest sample holding (about 25% of the money invested).